# Paper Analyst -- Example Notebook

Demonstrates the reproducibility pipeline's **Paper Analyst** (Agent 1).

Given a paper uploaded to NotebookLM and an optional DOI, produces a structured
`PaperProfile` JSON with methodology steps, datasets, figures, and repository links.

**Prerequisites:**
- `pip install pydantic requests`
- NotebookLM skill installed at `~/.claude/skills/notebooklm/` with auth set up
- Paper already uploaded to a NotebookLM notebook via the web UI

## Step 1: Authenticate to NotebookLM

Validates the stored browser session. If expired, opens a Chromium window for
Google login — sign in and the cell will continue automatically.

In [1]:
from pipeline.notebooklm_client import NotebookLMClient, NotebookLMError

client = NotebookLMClient()
try:
    client.connect()
    print("Session is live.")
except NotebookLMError:
    print("Session expired — opening browser for login...")
    client.authenticate(headless=False)
    print("Authenticated.")

Session is live.


## Step 2: Configure and run the analyst

`extract_profile()` connects to NotebookLM, runs 5 queries against the
uploaded paper, resolves the DOI via CrossRef, and parses everything into
a structured `PaperProfile`.

In [2]:
from pathlib import Path
from pipeline.analyst import AnalystConfig, extract_profile
from pipeline.ledger import Ledger

# -- Edit these for your paper --
config = AnalystConfig(
    notebook_url="https://notebooklm.google.com/notebook/46d04802-96e2-402a-8902-a7132c91cbbc",
    doi="10.1111/mice.13116",
    notebook_id="zhao-2023-gan",
    headless=True,
)

ledger = Ledger(path=Path("repro_ledger.jsonl"))
profile = extract_profile(config, ledger=ledger)

print(f"Title:    {profile.title}")
print(f"Authors:  {', '.join(profile.authors)}")
print(f"DOI:      {profile.doi}")
print(f"Steps:    {len(profile.methodology_steps)}")
print(f"Datasets: {len(profile.datasets)}")
print(f"Figures:  {len(profile.figures_to_reproduce)}")
print(f"Tables:   {len(profile.tables_to_reproduce)}")
print(f"Repos:    {profile.repository_links}")

Title:    Coloring and fusing architectural sketches by combining a Y‐shaped generative adversarial network and a denoising diffusion implicit model
Authors:  Liang Zhao, Dexuan Song, Weizhen Chen, Qi Kang
DOI:      10.1111/mice.13116
Steps:    8
Datasets: 6
Figures:  2
Tables:   1
Repos:    ['https://doi.org/10.1111/mice.13116', 'https://github.com/abc637725/muti-headed-Y-GAN', 'http://www.y-gan.net/', 'https://onlinelibrary.wiley.com/doi/10.1111/mice.13116', 'https://arxiv.org/abs/1701.07875', 'https://arxiv.org/abs/2010.02502', 'http://doi.org/10.48550/ARXIV.2006.11239']


## Step 3: Inspect methodology steps

In [3]:
for step in profile.methodology_steps:
    print(f"Step {step.order}: {step.description[:100]}")
    if step.tools_mentioned:
        print(f"  Tools:   {step.tools_mentioned}")
    if step.data_inputs:
        print(f"  Inputs:  {step.data_inputs}")
    if step.expected_outputs:
        print(f"  Outputs: {[o[:80] + '...' if len(o) > 80 else o for o in step.expected_outputs]}")
    print()

Step 1: Data Collection and Augmentation: The researchers collected a base set of architectural photos and e
  Inputs:  ['600 initial photos of low-rise buildings collected from the internet\nSoftware Tools/Libraries: Version not stated']
  Outputs: ['An augmented architectural dataset consisting of 10,000 images']

Step 2: Input Pre-processing and Standardization: Each color image is converted into a corresponding sketch 
  Tools:   ['XDoG']
  Inputs:  ['RGB architectural images from the augmented dataset\nSoftware Tools/Libraries: extended difference-of-Gaussians (XDoG) operator', 'version not stated']
  Outputs: ['A standardized dataset of paired 256 × 256 pixel sketches and RGB images']

Step 3: Neural Network Training: The Y-shaped GAN is trained for one day to minimize reconstruction, adversa
  Tools:   ['Adam', 'DDIM', 'GAN', 'U-Net', 'VGG-19']
  Inputs:  ['Standardized pairs of sketches and color images (for GAN training) and colorful architectural data (for DDIM training)\nSof

## Step 4: Inspect datasets

In [4]:
for ds in profile.datasets:
    print(f"  {ds.name}")
    print(f"    url:     {ds.url}")
    print(f"    licence: {ds.licence}")
    if ds.size_estimate:
        print(f"    size:    {ds.size_estimate}")
    print()

  
    url:     None
    licence: None
    size:    10,000 images (augmented from 600 original photos) [2].

  
    url:     None
    licence: None
    size:    10,000 images [2].

  
    url:     None
    licence: None
    size:    version not stated

  
    url:     None
    licence: None
    size:    50 categories (including motorcycles, horses, and couches) [8].

  
    url:     None
    licence: None
    size:    version not stated

  
    url:     None
    licence: None
    size:    version not stated



## Step 5: Export as JSON

Export the profile matching the target schema (without raw responses).

In [6]:
import json

# Export clean profile (exclude raw responses and internal fields)
profile_dict = profile.model_dump(
    exclude={"raw_analyst_responses", "stated_software_versions"},
    exclude_none=True,
)

profile_json = json.dumps(profile_dict, indent=2, ensure_ascii=False)

# Save to file
output_path = "paper_profile.json"
with open(output_path, "w") as f:
    f.write(profile_json)

print(f"Saved to {output_path} ({len(profile_json):,} bytes)")
print()
print(profile_json[:1500])
if len(profile_json) > 1500:
    print(f"  ... ({len(profile_json) - 1500:,} more chars)")

Saved to paper_profile.json (9,628 bytes)

{
  "doi": "10.1111/mice.13116",
  "title": "Coloring and fusing architectural sketches by combining a Y‐shaped generative adversarial network and a denoising diffusion implicit model",
  "authors": [
    "Liang Zhao",
    "Dexuan Song",
    "Weizhen Chen",
    "Qi Kang"
  ],
  "methodology_steps": [
    {
      "order": 1,
      "description": "Data Collection and Augmentation: The researchers collected a base set of architectural photos and expanded them into a larger dataset using common augmentation methods such as scaling, cutting, and mirror transition to ensure model robustness",
      "tools_mentioned": [],
      "data_inputs": [
        "600 initial photos of low-rise buildings collected from the internet\nSoftware Tools/Libraries: Version not stated"
      ],
      "expected_outputs": [
        "An augmented architectural dataset consisting of 10,000 images"
      ],
      "confidence": 0.7
    },
    {
      "order": 2,
      "descr